# Lighting, Color & Sampling

**Author:** Iman Mossavat · Fontys ICT

## What is this notebook about?

This notebook walks through the lowest-level building blocks used by the other experiments: raw noise, smoothing, sampling, palette mapping, lighting, and finishing. Understanding these pieces makes the other notebooks easier to read and modify.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from math_vis import smooth_field, sample, squash, make_cmap, cosine_cmap, upfield, finish, apply_glass_lighting, apply_lead_lines

SEED = 3
W, H = 300, 300
rng = np.random.default_rng(SEED)

## From noise to smooth fields

A small grid of random values is blurred with a Gaussian filter. The result is then upscaled to the full image size. Smoothing creates spatial structure; upscaling lets us work cheaply on a coarse field.

In [ ]:
# 1. Build a small random field and a full-size version
small = rng.normal(size=(40, 40))
field = smooth_field((40, 40), 8, rng)
big = upfield(field, W, H)

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(axs, [small, field, big], ['raw noise', 'smooth_field', 'upfield to full size']):
    im = ax.imshow(data, cmap='gray', origin='lower')
    ax.set_title(title); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

> **Question:** Why might we prefer to smooth a small field and then upscale it, rather than smoothing a full-size field?

<details><summary>Answer</summary>

DRAFT: author to review

`smooth_field` builds a small random grid and blurs it. Generating noise at full resolution and then blurring is much slower; generating a small field and upsampling with `upfield` gives the same large-scale structure with far fewer samples. The `mode='wrap'` blur also makes the field tileable, which is useful when the image is cropped or repeated.

</details>

## Sampling at fractional positions

`sample` uses bilinear interpolation to read the field at any fractional (row, col). Compare the upscaled field with the resampled version: sampling preserves smoothness without introducing resize artifacts.

In [ ]:
# 2. Sample the field at fractional positions
rows = np.linspace(0, 39, H)
cols = np.linspace(0, 39, W)
RR, CC = np.meshgrid(rows, cols, indexing='ij')
resampled = sample(field, RR, CC)

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].imshow(big, cmap='gray', origin='lower'); axs[0].set_title('PIL resize'); axs[0].axis('off')
axs[1].imshow(resampled, cmap='gray', origin='lower'); axs[1].set_title('map_coordinates sample'); axs[1].axis('off')
plt.tight_layout(); plt.show()

> **Question:** In what situations would `sample` give a visibly different result from a simple image resize?

<details><summary>Answer</summary>

DRAFT: author to review

`sample` uses bilinear interpolation (`map_coordinates` with `order=1`), so it blends the four nearest grid values. The result differs most where the field changes quickly between grid points — i.e. when the feature scale is small or the coordinates are displaced by a warp. For very smooth fields the interpolated value is close to the nearest grid value, so the difference is hard to see.

</details>

## Mapping values to [0,1]

`squash` is a soft sigmoid-like map. It lets unbounded field values become palette coordinates. The `softness` parameter controls how sharply the transition around zero happens.

In [ ]:
# 3. squash maps unbounded values to [0,1]
x = np.linspace(-4, 4, 500)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, squash(x), label='softness=1.5')
ax.plot(x, squash(x, softness=3.0), label='softness=3.0')
ax.axhline(0.5, color='gray', ls='--')
ax.set_title('squash: tanh-based map to [0,1]'); ax.legend(); plt.show()

> **Question:** A larger `softness` makes the curve flatter near zero. Would that increase or decrease contrast in the final image?

<details><summary>Answer</summary>

DRAFT: author to review

Larger `softness` stretches the tanh curve, so the same range of raw field values gets compressed into a narrower band around 0.5. Extremes still map near 0 and 1, but the mid-tones become more similar, reducing contrast in the centre of the palette range.

</details>

## Palettes

Two palette builders are shown: one from a list of hex colors, and one from cosine parameters. Both turn a scalar field into an RGB image.

In [ ]:
# 4. Palettes
hex_cmap = make_cmap(['#0b1d3a', '#1f4e8c', '#58d3c0', '#f4e285', '#e0453a'])
cos_cmap = cosine_cmap(a=(0.6, 0.55, 0.62), b=(0.38, 0.38, 0.34), c=(0.85, 0.85, 0.85), d=(0.0, 0.30, 0.62))

fig, axs = plt.subplots(2, 1, figsize=(10, 4))
t = np.linspace(0, 1, 256)
axs[0].imshow(hex_cmap(t)[None, :, :3], aspect='auto'); axs[0].set_title('make_cmap from hex list'); axs[0].set_yticks([])
axs[1].imshow(cos_cmap(t)[None, :, :3], aspect='auto'); axs[1].set_title('cosine_cmap: a + b*cos(2π(c*t+d))'); axs[1].set_yticks([])
plt.tight_layout(); plt.show()

> **Question:** The cosine palette can be shifted by changing `d`. What happens to the colors if you add 0.5 to each component of `d`?

<details><summary>Answer</summary>

DRAFT: author to review

In Inigo Quilez's cosine palette `color(t) = a + b * cos(2π * (c*t + d))`, the `d` vector shifts the phase of the cosine for each channel. Changing `d` rotates the hue cycle, so the same field value maps to a different colour; warm highlights can become cool highlights and vice versa.

</details>

## From palette to colored field

Here the same smooth field is mapped through both palettes. The structure is identical; only the color recipe changes.

In [ ]:
# 5. Map a smooth field to colors
color_field = smooth_field((60, 60), 10, rng)
tmap = squash(upfield(color_field, W, H))
img_hex = hex_cmap(tmap)[:, :, :3] * 255
img_cos = cos_cmap(tmap)[:, :, :3] * 255

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(img_hex.astype(np.uint8)); axs[0].set_title('hex palette'); axs[0].axis('off')
axs[1].imshow(img_cos.astype(np.uint8)); axs[1].set_title('cosine palette'); axs[1].axis('off')
plt.tight_layout(); plt.show()

> **Question:** If the field had no smoothness (pure noise), would you still see large coherent color regions?

<details><summary>Answer</summary>

DRAFT: author to review

No. Without smoothing the field values are independent per pixel, so mapping them through the palette gives a noisy, salt-and-pepper colour pattern. The large coherent regions come entirely from the spatial correlation introduced by the Gaussian blur; the palette only colours an already smooth structure.

</details>

## Lighting

`apply_glass_lighting` darkens cell edges and adds a glow, while `apply_lead_lines` draws borders between neighboring cells. These effects turn flat color into the appearance of material.

In [ ]:
# 6. Lighting on a few synthetic cells
from scipy.spatial import cKDTree
n = 12
seeds = rng.uniform([0, 0], [W, H], size=(n, 2))
X, Y = np.meshgrid(np.arange(W)+0.5, np.arange(H)+0.5)
P = np.column_stack([X.ravel(), Y.ravel()])
d, idx = cKDTree(seeds).query(P, k=2)
cell = idx[:, 0]
t_cell = rng.uniform(0, 1, n)
flat = hex_cmap(t_cell[cell])[:, :3] * 255
lit = apply_glass_lighting(flat, P, seeds, d[:, 0], cell, facet=0.6, glow=0.5, rng=rng)
leaded = apply_lead_lines(lit, d[:, 0], d[:, 1], W, H, lead_px=3, rng=rng)

fig, axs = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, img, title in zip(axs, [flat, lit, leaded], ['flat color', '+ facet/glow lighting', '+ lead lines']):
    ax.imshow(np.clip(img, 0, 255).reshape(H, W, 3).astype(np.uint8))
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

> **Question:** Which parameter — `facet` or `glow` — controls how strongly the cell interiors look rounded?

<details><summary>Answer</summary>

DRAFT: author to review

`glow` controls the rounded, glassy look of the cell interiors. It multiplies brightness by `0.45 - d1/R`, which is brightest at the cell centre and falls off toward the lead lines. `facet` controls directional tilt and adds the appearance of a cut facet, but the rounded volume effect is mainly `glow`.

</details>

## Finishing

Finally, grain and vignette add photographic texture and draw the eye toward the center.

In [ ]:
# 7. Finishing effects: grain and vignette
polished = finish(leaded.reshape(H, W, 3), grain=3, vignette=0.4, display=False)
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(np.clip(leaded,0,255).reshape(H,W,3).astype(np.uint8)); axs[0].set_title('before finish'); axs[0].axis('off')
axs[1].imshow(polished); axs[1].set_title('after finish (grain + vignette)'); axs[1].axis('off')
plt.tight_layout(); plt.show()

> **Question:** Does `vignette=0` make the image feel more like a digital graphic or a photograph?

<details><summary>Answer</summary>

DRAFT: author to review

`vignette=0` removes the edge darkening, so the image has uniform brightness all the way to the border. That feels more like a clean digital graphic. A positive vignette mimics the light falloff of a camera lens or old print, making the result feel more like a photograph.

</details>

## What did we learn?

- Smoothing turns noise into fields; sampling maps those fields to new resolutions.
- `squash` remaps unbounded values to `[0,1]`.
- Palettes turn scalar fields into color images.
- Lighting and finishing add material and photographic quality.

**Open questions:**
1. Why does `map_coordinates` produce a different look than a simple resize?
2. How does `softness` in `squash` affect contrast?

**Further experiment:** Combine the hex palette with the cosine palette by mixing their RGB values and map the result back to a colored field.

## Further reading

- Conceptual background: [From randomness to structure](conceptualGuide.md#1-from-randomness-to-structure) and [When numbers become motion](conceptualGuide.md#2-when-numbers-become-motion) in `conceptualGuide.md`.
- Implementation details: [Random scalar fields](recipes.md#1-random-scalar-fields), [Palettes and color mapping](recipes.md#2-palettes-and-color-mapping), and [Glass lighting model](recipes.md#8-glass-lighting-model-glass) in `recipes.md`.